In [5]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.decomposition import PCA

In [27]:
def compute_mmd(X, Y, kernel='rbf', gamma=0.01):
    """
    Compute the Maximum Mean Discrepancy (MMD) between two  feature datasets X(trained data) and Y (new data).
    """
    # Compute kernels
    K_XX = rbf_kernel(X, X, gamma=gamma)
    K_YY = rbf_kernel(Y, Y, gamma=gamma)
    K_XY = rbf_kernel(X, Y, gamma=gamma)
    
    n = X.shape[0]
    m = Y.shape[0]
    
    # Remove diagonal elements from K_XX and K_YY
    np.fill_diagonal(K_XX, 0)
    np.fill_diagonal(K_YY, 0)
    
    mmd = (K_XX.sum() / (n * (n - 1))) + (K_YY.sum() / (m * (m - 1))) - (2 * K_XY.mean())
    return mmd

In [38]:
def shift_detection(X, Y, n_components, kernel='rbf', gamma=0.01, threshold=0.01):
    pca = PCA(n_components=n_components)  # Use n_components to control the number of principal components
    
    X_pca = pca.fit_transform(X)
    Y_pca = pca.transform(Y)  # Transform Y with the same PCA object
    
    # Compute the MMD score between X and Y using the specified kernel and gamma
    mmd_score = compute_mmd(X_pca, Y_pca, kernel, gamma)
    
    # Check if the MMD score exceeds the threshold
    if mmd_score > threshold:
        print(f"Shift detected! Maximum Mean Discrepancy (MMD) score: {mmd_score:.4f} exceeds threshold of {threshold}")
    else:
        print(f"No significant shift detected. Maximum Mean Discrepancy (MMD) score: {mmd_score:.4f} is within threshold.")

In [39]:
# Preprocess Datasets for better mmd computation
X_train = pd.read_csv("../data/X_train.csv")
X_new = pd.read_csv("../data/X_test_2.csv")

shift_detection(X_train, X_new, 50)

Shift detected! Maximum Mean Discrepancy (MMD) score: 0.0118 exceeds threshold of 0.01
